<a href="https://colab.research.google.com/github/Abraham-Na70/B401-Cenim/blob/main/module/module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modul Praktikum: Human-Robot Interaction Workshop
**Laboratorium B401: Sistem Tertanam dan Robotika**

Selamat datang di workshop interaksi manusia dan robot (Human-Robot Interaction: HRI). Dalam modul praktikum terpadu ini, Anda akan mempelajari bagaimana menjembatani persepsi manusia (gerak tubuh, gestur tangan, dan perintah bahasa alami) dengan aktuasi fisik sistem robotika modern:
1. **Lengan Robot Industri (Universal Robots UR5)** untuk tugas manipulasi presisi tinggi (Pick and Place).
2. **Robot Berkaki Empat (Quadruped RoboDog)** untuk tugas navigasi dinamis dan mobilitas lingkungan.

---

### Alur Kerja Praktikum:
* **Bab 0: Persiapan Lingkungan dan Protokol Komunikasi Multi-Robot**
  Setup pustaka, inisialisasi sesi komunikasi asinkron, dan manajemen antrean sewa eksklusif (Time-Leased Mutex Lock).
* **Bab 1: Persepsi Visual dan Rekayasa Fitur Pose-Posture (Computer Vision & Machine Learning)**
  Ekstraksi 21 koordinat sendi tangan menggunakan MediaPipe, normalisasi 42-fitur spasial invarian, studio perekaman dataset, dan klasifikasi Random Forest.
* **Bab 2: Perintah Kompleks Lengan Robot UR5 Berbasis LLM (Large Language Model)**
  Penerjemahan perintah bahasa alami manusia menjadi rencana aksi terstruktur menggunakan rekayasa prompt dan reasoning LLM.
* **Bab 3: Kendali Teleoperasi Waypoint Lengan Robot Manipulator UR5**
  Pemetaan koordinat papan catur (A1-H8), filter kestabilan dwell debouncer (1.0 detik), dan kendali teleoperasi live dari kamera peramban.
* **Bab 4: Interaksi dan Kendali Robot Berkaki Empat (Quadruped RoboDog)**
  Koneksi API robot, pemetaan pose tubuh ke respon gerak/stance RoboDog, serta skenario interaksi dinamis.


# Bab 0: Persiapan Lingkungan dan Komunikasi Sistem

## 0.1 Konteks Konseptual: Arsitektur Komunikasi Terdistribusi

Dalam workshop HRI ini, komputer lab yang terhubung ke perangkat keras robot bertindak sebagai **Server Kendali**. Sementara itu, Google Colab yang Anda jalankan bertindak sebagai **Klien Pengendali**.

Karena komputer lab berada di balik jaringan lokal kampus, kita menggunakan terowongan **Cloudflare Tunnel** (`wss://ur5.fawzqi.my.id`). Protokol WebSocket dipilih karena memiliki latensi rendah dan mendukung komunikasi dua arah secara asinkron.

### Mengapa Kita Menggunakan Sistem Sewa Kunci (Time-Leased Mutex Lock)?

Dalam satu sesi praktikum, puluhan mahasiswa menjalankan notebook ini secara bersamaan. Jika beberapa mahasiswa mengirim perintah gerak ke satu robot fisik pada detik yang sama, robot akan menerima instruksi yang bertentangan dan dapat memicu penghentian darurat (protective stop).

Untuk mencegah tabrakan perintah (*race condition*), server lab menerapkan mekanisme **Time-Leased Mutex Lock**:
* Mahasiswa yang ingin menggerakkan robot wajib meminta token sewa eksklusif (`acquire_lock`).
* Selama token aktif (misalnya 45 detik), server hanya menerima perintah dari mahasiswa pemegang token.
* Jika mahasiswa selesai lebih awal, kunci dapat dilepas (`release_lock`). Jika mahasiswa terputus koneksinya, kunci akan kedaluwarsa secara otomatis agar giliran mahasiswa lain tidak tertahan.


In [ ]:
#@title 0.1 Instalasi Dependensi Pustaka dan Unduh Model Deteksi Tangan { display-mode: "form" }
# 0.1 Instalasi Dependensi Pustaka dan Unduh Model Deteksi Tangan
# Harap jangan mengubah isi sel ini

import os
import sys

print("[1/2] Memeriksa dan menginstal pustaka yang diperlukan...")
!pip install -q websockets mediapipe opencv-python-headless scikit-learn matplotlib joblib

MODEL_PATH = "hand_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

print("[2/2] Memeriksa ketersediaan file model MediaPipe Hand Landmarker...")
if not os.path.exists(MODEL_PATH):
    print(f"Mengunduh model dari {MODEL_URL}...")
    !wget -q -O {MODEL_PATH} {MODEL_URL}
    print("Model berhasil diunduh.")
else:
    print("File model sudah tersedia di direktori kerja.")

print("Instalasi selesai dan lingkungan siap digunakan.")


In [ ]:
#@title 0.2 Modul Klien Sesi Robot (RobotSession) { display-mode: "form" }
# Harap jangan mengubah isi sel ini

import asyncio
import json
import time
import websockets

def _msg(msg_type: str, payload: dict) -> str:
    """Helper serializer envelope standar untuk komunikasi dengan robot_server."""
    return json.dumps({
        "msg_id": str(int(time.time() * 1000) % 10**8),
        "msg_type": msg_type,
        "timestamp": time.time(),
        "payload": payload,
    })

class RobotSession:
    """
    Klien komunikasi asinkron ke server Lab PC.
    Mengelola siklus sewa mutex lock dan eksekusi rencana gerak robot.
    """
    def __init__(self, ws_url: str, student_id: str):
        self.ws_url = ws_url
        self.student_id = student_id
        self.token = None

    async def ping(self) -> dict:
        """Mengirim pesan heartbeat untuk menguji konektivitas server."""
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                await ws.send(_msg("HEARTBEAT", {"client": self.student_id}))
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                return json.loads(raw).get("payload", {})
        except Exception as e:
            return {"error": str(e)}

    async def acquire_lock(self, duration_s: float = 45.0) -> bool:
        """Meminta token sewa eksklusif kendali robot."""
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                req = _msg("ACQUIRE_LEASE", {"student_id": self.student_id, "duration_s": duration_s})
                await ws.send(req)
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                data = json.loads(raw).get("payload", {})
                if data.get("granted"):
                    self.token = data.get("token")
                    print(f"[LEASING] Kunci kendali berhasil diperoleh untuk '{self.student_id}'. Berlaku {duration_s} detik.")
                    return True
                else:
                    holder = data.get("holder", "kelompok lain")
                    wait_time = data.get("wait_seconds", "?")
                    print(f"[LEASING] Robot sedang digunakan oleh '{holder}'. Sisa waktu sewa: {wait_time} detik.")
                    return False
        except Exception as e:
            print(f"[LEASING ERROR] Gagal menghubungi server pengelola sewa: {e}")
            return False

    async def release_lock(self) -> bool:
        """Melepaskan token sewa agar robot dapat digunakan oleh kelompok lain."""
        if not self.token:
            return True
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                req = _msg("RELEASE_LEASE", {"student_id": self.student_id, "token": self.token})
                await ws.send(req)
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                data = json.loads(raw).get("payload", {})
                self.token = None
                print(f"[LEASING] Kunci kendali '{self.student_id}' telah dilepaskan secara aman.")
                return data.get("released", True)
        except Exception as e:
            print(f"[LEASING ERROR] Terjadi kendala saat melepaskan kunci: {e}")
            self.token = None
            return False

    async def execute(self, action: str) -> bool:
        """Mengirim perintah aksi tunggal ke robot fisik/simulasi."""
        plan_id = f"plan_{int(time.time() * 1000) % 10**6}"
        payload = {
            "plan_id": plan_id,
            "actions": [action],
            "parameters": {"token": self.token, "client": self.student_id},
        }

        try:
            async with websockets.connect(self.ws_url, open_timeout=10.0) as ws:
                await ws.send(_msg("PLAN_DISPATCH", payload))
                ack_raw = await asyncio.wait_for(ws.recv(), timeout=8.0)
                ack = json.loads(ack_raw).get("payload", {})
                if not ack.get("accepted"):
                    print(f"[DISPATCH DITOLAK] {ack.get('message', 'Perintah tidak diterima oleh server')}")
                    return False

                print(f"[STATUS: RUNNING] Menjalankan perintah [{action}]...")
                while True:
                    raw = await asyncio.wait_for(ws.recv(), timeout=35.0)
                    msg = json.loads(raw)
                    mtype = msg.get("msg_type")
                    p = msg.get("payload", {})

                    if mtype == "STEP_PROGRESS":
                        if p.get("status") == "SUCCESS":
                            elapsed = p.get("elapsed_s", 0.0)
                            print(f"[STATUS: SUCCESS] Selesai dalam {elapsed:.2f} detik.")
                        elif p.get("status") == "FAILED":
                            print(f"[STATUS: FAILED] Eksekusi langkah gagal: {p.get('message')}")
                            return False

                    elif mtype == "PLAN_COMPLETE":
                        status = p.get("status")
                        return status == "COMPLETED"

        except Exception as e:
            print(f"[EKSEKUSI ERROR] Koneksi terputus saat eksekusi [{action}]: {e}")
            return False

print("Modul klien RobotSession siap digunakan.")


## 0.2 Konfigurasi Identitas Kelompok dan URL Server

Silakan sesuaikan konfigurasi di bawah ini dengan identitas kelompok Anda. URL WebSocket mengarah ke terowongan Cloudflare Lab PC yang aktif.


In [ ]:
# --- TULIS KONFIGURASI KELOMPOK DI SINI ---
STUDENT_ID = "Kelompok_01"
ROBOT_TUNNEL_WS = "wss://ur5.fawzqi.my.id"
# -------------------------------------------

session = RobotSession(ws_url=ROBOT_TUNNEL_WS, student_id=STUDENT_ID)
print(f"Klien sesi diinisialisasi untuk: {STUDENT_ID}")
print(f"Target Server: {ROBOT_TUNNEL_WS}")


## 0.3 Pengujian Konektivitas dan Jabat Tangan Awal

Jalankan sel berikut untuk memverifikasi apakah Google Colab dapat berkomunikasi dengan server Lab PC. Sel ini akan melakukan tes ping (heartbeat) dan mengeksekusi satu rencana waypoint uji coba.


In [ ]:
async def verify_initial_connection():
    print("[1/3] Mengirim sinyal ping ke server lab...")
    status = await session.ping()
    if "error" in status:
        print(f"[PERINGATAN] Gagal terhubung ke server: {status['error']}")
        print("Pastikan server dan terowongan Cloudflare di lab PC dalam keadaan aktif.")
        return False

    print(f"[2/3] Terhubung dengan baik. Status sewa server: {status.get('lease_status')}")
    
    print(f"[3/3] Meminta kunci kendali untuk uji coba awal...")
    has_lock = await session.acquire_lock(duration_s=30.0)
    if not has_lock:
        print("Kunci sedang digunakan oleh kelompok lain. Silakan coba kembali sesaat lagi.")
        return False

    print("Mengirim perintah uji coba: [READY]...")
    ok = await session.execute("READY")
    await session.release_lock()
    
    if ok:
        print("Pengujian awal Bab 0 berhasil. Seluruh dependensi dan jaringan siap.")
    return ok

await verify_initial_connection()


### Apa yang Kita Pelajari dari Output Ini?
1. **Verifikasi Jalur Komunikasi:** Pesan JSON berhasil melewati terowongan Cloudflare dan diterima oleh server tanpa kendala firewall atau izin port.
2. **Validasi Mutex Lock:** Server memvalidasi token unik sebelum mengizinkan perintah dijalankan, membuktikan bahwa sistem aman dari tabrakan instruksi.
3. **Telemetri Eksekusi:** Server mengembalikan konfirmasi waktu eksekusi langkah secara terukur, menandakan bahwa modul kendali di lab siap menerima perintah otomatis.


# Bab 1: Persepsi Visual dan Rekayasa Fitur Pose-Posture (Computer Vision & Machine Learning)

## 1.1 Konteks Konseptual: Mengapa Fitur Geometri Landmark?

Tujuan Bab 1 adalah membangun sistem cerdas yang mampu mengenali pose tangan dan postur tubuh pengguna secara real-time dari kamera peramban.

### Masalah pada Citra Piksel Mentah
Jika kita melatih model pembelajaran mesin langsung menggunakan piksel gambar mentah:
* Satu gambar berukuran $640 \times 480$ dengan 3 kanal warna (RGB) menghasilkan **921.600 angka**.
* Model akan sangat mudah terpengaruh oleh warna dinding latar belakang, pencahayaan ruangan, warna kulit, dan jarak tangan ke kamera (*overfitting* parah).
* Membutuhkan waktu pelatihan yang lama dan sumber daya komputasi tinggi.

### Solusi: Ekstraksi 21 Keypoints MediaPipe dan Normalisasi 42-Fitur
MediaPipe Hands menggunakan arsitektur jaringan saraf konvolusi terpisah untuk melokalisasi 21 titik persendian tangan (tulang jari) dalam format koordinat $(x, y)$.

Agar koordinat tersebut tidak berubah saat tangan bergeser ke kiri/kanan layar atau mendekat/menjauh dari kamera, kita menerapkan **Normalisasi Spasial Invarian**:
1. **Invarian Translasi:** Menggeser titik pergelangan tangan (*wrist*, titik indeks 0) ke titik asal $(0, 0)$. Semua titik lainnya dikurangi dengan koordinat pergelangan tangan.
2. **Invarian Skala:** Membagi seluruh koordinat relatif dengan rentang telapak tangan, yaitu jarak antara pergelangan tangan (titik 0) dan pangkal jari tengah (*middle MCP*, titik 9).

Rumus normalisasi matematis untuk setiap titik $p_i = (x_i, y_i)$:

\begin{equation}
p_{\text{rel}, i} = p_i - p_{\text{wrist}}
\end{equation}

\begin{equation}
s = \| p_{\text{middle\_mcp}} - p_{\text{wrist}} \|
\end{equation}

\begin{equation}
p_{\text{norm}, i} = \frac{p_{\text{rel}, i}}{s}
\end{equation}

Hasil akhirnya adalah **vektor 42 angka float** (21 titik $\times$ 2 dimensi) yang sepenuhnya independen dari ukuran tangan, posisi tangan di layar, maupun latar belakang ruangan.


In [ ]:
#@title 1.1 Inisialisasi Detektor Landmark Tangan (Universal MediaPipe) { display-mode: "form" }
# Harap jangan mengubah isi sel ini

import os
import cv2
import numpy as np
import mediapipe as mp

class HandResult:
    """Wrapper untuk menyelaraskan struktur keluaran MediaPipe modern dan legacy."""
    def __init__(self, landmarks):
        self.multi_hand_landmarks = [landmarks] if landmarks is not None else None

class UniversalHandDetector:
    """
    Detektor tangan fleksibel:
    Mendukung MediaPipe modern (>=0.10.30, Tasks API) maupun legacy (mp.solutions.hands).
    """
    def __init__(self, model_path: str = "hand_landmarker.task"):
        self.mode = "legacy"
        if hasattr(mp, "solutions") and hasattr(mp.solutions, "hands"):
            self.detector = mp.solutions.hands.Hands(
                static_image_mode=True,
                max_num_hands=1,
                min_detection_confidence=0.6,
                min_tracking_confidence=0.6,
            )
        else:
            self.mode = "tasks"
            if not os.path.exists(model_path):
                import urllib.request
                url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
                print(f"Mengunduh model MediaPipe Tasks ({model_path})...")
                urllib.request.urlretrieve(url, model_path)

            from mediapipe.tasks import python
            from mediapipe.tasks.python import vision

            base_options = python.BaseOptions(model_asset_path=model_path)
            options = vision.HandLandmarkerOptions(
                base_options=base_options,
                running_mode=vision.RunningMode.IMAGE,
                num_hands=1,
                min_hand_detection_confidence=0.6,
                min_tracking_confidence=0.6,
            )
            self.detector = vision.HandLandmarker.create_from_options(options)

    def detect_landmarks(self, rgb_image: np.ndarray):
        """Menerima citra RGB (H, W, 3), mengembalikan 21 titik koordinat landmark tangan."""
        if self.mode == "legacy":
            results = self.detector.process(rgb_image)
            if results.multi_hand_landmarks:
                return results.multi_hand_landmarks[0]
            return None
        else:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
            results = self.detector.detect(mp_image)
            if results.hand_landmarks and len(results.hand_landmarks) > 0:
                return results.hand_landmarks[0]
            return None

    def process(self, rgb_image: np.ndarray):
        """Metode antarmuka yang selaras dengan mp.solutions.hands.process()."""
        lms = self.detect_landmarks(rgb_image)
        return HandResult(lms)


hands_detector = UniversalHandDetector()

GESTURE_LABELS = [
    "POINT",
    "PEACE",
    "THREE_FINGERS",
    "FOUR_FINGERS",
    "OK_SIGN",
    "OPEN_PALM",
    "THUMBS_UP",
    "FIST",
]

def extract_normalized_features(landmarks) -> np.ndarray:
    """
    Mengubah 21 titik landmark MediaPipe menjadi vektor 42 fitur spasial invarian:
      1. Translasi: menggeser titik pergelangan (titik 0) ke (0, 0).
      2. Skala: membagi koordinat dengan jarak pergelangan ke pangkal jari tengah (titik 9).
    """
    if landmarks is None:
        return None
    lms = getattr(landmarks, "landmark", landmarks)
    coords = np.array([[lm.x, lm.y] for lm in lms])
    if len(coords) < 21:
        return None

    wrist = coords[0]
    translated = coords - wrist

    palm_span = np.linalg.norm(translated[9])
    if palm_span < 1e-6:
        palm_span = 1.0
    normalized = translated / palm_span

    return normalized.flatten()

print(f"Universal Hand Detector ({hands_detector.mode.upper()} mode) dan fungsi ekstraksi 42-fitur siap.")


## 1.2 Pengumpulan Dataset Gesture Tangan

Kita akan mengumpulkan dataset untuk melatih model mengenali **8 gesture kendali dasar**. Berikut adalah pemetaan peran masing-masing gesture:

| Nama Gesture | Deskripsi Pose Tangan | Peran Kendali Sistem Robot |
| :--- | :--- | :--- |
| `POINT` | Mengacungkan 1 jari (telunjuk) | Navigasi Posisi 1 / Target A |
| `PEACE` | Mengacungkan 2 jari (telunjuk dan tengah) | Navigasi Posisi 2 / Target B |
| `THREE_FINGERS` | Mengacungkan 3 jari | Navigasi Posisi 3 / Target C |
| `FOUR_FINGERS` | Mengacungkan 4 jari | Navigasi Posisi 4 / Target D |
| `OK_SIGN` | Menyatukan ujung telunjuk dan jempol (lingkaran) | Perintah Interaksi 1 (Ambil Benda / Stance Duduk) |
| `OPEN_PALM` | Membuka seluruh telapak tangan (5 jari) | Perintah Interaksi 2 (Lepas Benda / Stance Berdiri) |
| `THUMBS_UP` | Mengacungkan jempol ke atas | Kembali ke Posisi Siap Netral (`HOME` / `READY`) |
| `FIST` | Mengepalkan seluruh jari tangan | Mode Aman / Tahan Diam (`HOLD` / `NEUTRAL`) |

---

### Petunjuk Penggunaan Studio Perekaman:
1. Jalankan sel kode di bawah. Dashboard interaktif akan muncul.
2. Izinkan akses kamera peramban jika muncul permintaan izin (*Camera: Allow*).
3. **Pilihan Mode Rekam:**
   * **Ganti Ulang (Replace) [Bawaan]:** Jika rekaman sebelumnya kurang pas atau buram, menekan tombol gesture yang sama akan menghapus 10 sampel lama dan menggantinya dengan yang baru.
   * **Tambah (Append):** Jika ingin menambah variasi sudut kemiringan tangan (misalnya 10 sampel menghadap lurus + 10 sampel sedikit miring), pilih mode ini.
4. Klik tombol gesture yang ingin direkam, perhatikan hitung mundur 3 detik, lalu tahan pose tangan di depan kamera selama 10 frame burst diambil.
5. Tombol **Muat Baseline Sintetis** dapat digunakan sebagai opsi instan jika perangkat Anda tidak memiliki webcam fisik yang berfungsi.


In [ ]:
#@title 1.2 Studio Perekaman Dataset Gesture (Webcam Browser) { display-mode: "form" }
# Harap jangan mengubah isi sel ini
# 1.2 Pengumpulan Dataset Gesture: Dashboard Interaktif Grid 8-Gesture
import time
import base64
import numpy as np
import cv2
from google.colab import output
from IPython.display import JSON, Javascript, display

DATASET_X = []
DATASET_Y = []

GESTURE_CLASSES = [
    ("POINT", "POINT (1 Jari)"),
    ("PEACE", "PEACE (2 Jari)"),
    ("THREE_FINGERS", "THREE (3 Jari)"),
    ("FOUR_FINGERS", "FOUR (4 Jari)"),
    ("OK_SIGN", "OK_SIGN (Ambil)"),
    ("OPEN_PALM", "OPEN_PALM (Lepas)"),
    ("THUMBS_UP", "THUMBS_UP (Home)"),
    ("FIST", "FIST (Netral/Hold)"),
]

# ── A. Callback Python: Memproses Burst Frames dengan Mode Replace/Append ──
def process_burst_payload(gesture_label, frames_b64, mode="replace"):
    """
    Menerima frame dari JS.
    mode='replace': Menghapus rekaman lama untuk gesture ini sebelum menambah yang baru.
    mode='append': Menambah pada rekaman yang sudah ada.
    """
    global DATASET_X, DATASET_Y

    if mode == "replace":
        filtered_x = [x for x, y in zip(DATASET_X, DATASET_Y) if y != gesture_label]
        filtered_y = [y for y in DATASET_Y if y != gesture_label]
        DATASET_X = filtered_x
        DATASET_Y = filtered_y

    valid_count = 0
    for b64_str in frames_b64:
        try:
            header, encoded = b64_str.split(',', 1) if ',' in b64_str else ('', b64_str)
            img_bytes = base64.b64decode(encoded)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
            if frame is None:
                continue
            rgb = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)

            lms = hands_detector.detect_landmarks(rgb)
            if lms is not None:
                feats = extract_normalized_features(lms)
                if feats is not None:
                    DATASET_X.append(feats)
                    DATASET_Y.append(gesture_label)
                    valid_count += 1
        except Exception:
            continue

    counts = {g[0]: DATASET_Y.count(g[0]) for g in GESTURE_CLASSES}
    total = len(DATASET_Y)
    return JSON({"valid": valid_count, "total": total, "counts": counts})

output.register_callback("notebook.ProcessBurstFrames", process_burst_payload)

# ── B. Reset Dataset Callback ──
def py_reset_dataset():
    global DATASET_X, DATASET_Y
    DATASET_X.clear()
    DATASET_Y.clear()
    counts = {g[0]: 0 for g in GESTURE_CLASSES}
    return JSON({"total": 0, "counts": counts})

output.register_callback("notebook.ResetDataset", py_reset_dataset)

# ── C. Fallback: Generator Data Sintetis ──
def py_load_synthetic():
    global DATASET_X, DATASET_Y
    DATASET_X.clear()
    DATASET_Y.clear()

    templates = {
        "FIST": [[0, 0]] + [[0.05 * i, -0.2] for i in range(1, 5)] + [[0.0, -0.35], [0.0, -0.3], [0.0, -0.25], [0.0, -0.2]] * 4,
        "POINT": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[0.0, -0.4], [0.0, -0.65], [0.0, -0.85], [0.0, -1.05]] + [[0.1, -0.35], [0.1, -0.3], [0.1, -0.25], [0.1, -0.2]] * 3,
        "PEACE": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[-0.08, -0.4], [-0.12, -0.65], [-0.15, -0.85], [-0.18, -1.05]] + [[0.08, -0.4], [0.12, -0.65], [0.15, -0.85], [0.18, -1.05]] + [[0.1, -0.35], [0.1, -0.3], [0.1, -0.25], [0.1, -0.2]] * 2,
        "THREE_FINGERS": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[-0.15, -0.4], [-0.2, -0.65], [-0.25, -0.85], [-0.3, -1.05]] + [[0.0, -0.4], [0.0, -0.65], [0.0, -0.85], [0.0, -1.05]] + [[0.15, -0.4], [0.2, -0.65], [0.25, -0.85], [0.3, -1.05]] + [[0.2, -0.35], [0.2, -0.3], [0.2, -0.25], [0.2, -0.2]],
        "FOUR_FINGERS": [[0, 0]] + [[-0.05, -0.1], [-0.08, -0.15], [-0.1, -0.2], [-0.12, -0.22]] + [[-0.2, -0.4], [-0.25, -0.65], [-0.3, -0.85], [-0.35, -1.05]] + [[-0.07, -0.4], [-0.09, -0.65], [-0.1, -0.85], [-0.12, -1.05]] + [[0.07, -0.4], [0.09, -0.65], [0.1, -0.85], [0.12, -1.05]] + [[0.2, -0.4], [0.25, -0.65], [0.3, -0.85], [0.35, -1.05]],
        "OPEN_PALM": [[0, 0]] + [[-0.25, -0.2], [-0.35, -0.3], [-0.45, -0.4], [-0.55, -0.5]] + [[-0.2, -0.4], [-0.25, -0.65], [-0.3, -0.85], [-0.35, -1.05]] + [[-0.07, -0.4], [-0.09, -0.65], [-0.1, -0.85], [-0.12, -1.05]] + [[0.07, -0.4], [0.09, -0.65], [0.1, -0.85], [0.12, -1.05]] + [[0.2, -0.4], [0.25, -0.65], [0.3, -0.85], [0.35, -1.05]],
        "THUMBS_UP": [[0, 0]] + [[-0.1, -0.3], [-0.15, -0.55], [-0.2, -0.8], [-0.25, -1.05]] + [[0.05, -0.25], [0.05, -0.2], [0.05, -0.15], [0.05, -0.1]] * 4,
        "OK_SIGN": [[0, 0]] + [[-0.1, -0.2], [-0.12, -0.3], [-0.1, -0.4], [-0.05, -0.45]] + [[0.0, -0.2], [0.0, -0.3], [0.0, -0.4], [-0.05, -0.45]] + [[0.1, -0.4], [0.12, -0.65], [0.15, -0.85], [0.18, -1.05]] + [[0.2, -0.4], [0.22, -0.65], [0.25, -0.85], [0.28, -1.05]] + [[0.3, -0.35], [0.32, -0.55], [0.35, -0.75], [0.38, -0.95]],
    }

    np.random.seed(42)
    for label, base_coords in templates.items():
        base = np.array(base_coords[:21])
        for _ in range(40):
            noise = np.random.normal(0, 0.025, base.shape)
            jittered = base + noise
            palm_span = np.linalg.norm(jittered[9]) or 1.0
            feats = (jittered / palm_span).flatten()
            DATASET_X.append(feats)
            DATASET_Y.append(label)

    counts = {g[0]: DATASET_Y.count(g[0]) for g in GESTURE_CLASSES}
    return JSON({"total": len(DATASET_Y), "counts": counts})

output.register_callback("notebook.LoadSyntheticBaseline", py_load_synthetic)

# ── D. Dashboard UI Terpercaya: Auto Hardware Track Release & Resilient Perms ──
DASHBOARD_JS = """
async function initDatasetStudio() {
  // 1. Release ANY orphaned camera stream from prior runs
  if (window.activeDatasetStream) {
    try { window.activeDatasetStream.getTracks().forEach(t => t.stop()); } catch(e){}
    window.activeDatasetStream = null;
  }
  if (window.teleopStream) {
    try { window.teleopStream.getTracks().forEach(t => t.stop()); } catch(e){}
    window.teleopStream = null;
  }

  const existing = document.getElementById('gesture-dashboard-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'gesture-dashboard-box';
  container.style.fontFamily = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif';
  container.style.background = '#1e1e1e';
  container.style.color = '#fff';
  container.style.padding = '18px';
  container.style.borderRadius = '12px';
  container.style.maxWidth = '680px';
  container.style.boxShadow = '0 4px 14px rgba(0,0,0,0.5)';
  container.style.marginTop = '10px';

  container.innerHTML = `
    <div style="display:flex; justify-content:space-between; align-items:center; border-bottom: 1px solid #333; padding-bottom: 10px; margin-bottom: 12px;">
      <div>
        <h3 style="margin:0; font-size: 18px; color:#00ffcc;">Studio Pengumpulan Dataset Gesture Tangan</h3>
        <div style="font-size: 12px; color: #aaa; margin-top:3px;">Pilih gesture -> hitung mundur 3s -> rekam 10 frame burst.</div>
      </div>
      <div id="dash-total-badge" style="background:#0984e3; padding: 4px 10px; border-radius: 6px; font-weight:bold; font-size:13px;">Total: 0 Sampel</div>
    </div>

    <!-- Mode Selector & Controls -->
    <div style="display:flex; justify-content:space-between; align-items:center; background:#2a2a2a; padding:6px 12px; border-radius:6px; margin-bottom:12px; font-size:12px;">
      <div style="display:flex; align-items:center; gap:12px;">
        <span>Mode Rekam:</span>
        <label style="cursor:pointer;"><input type="radio" name="recMode" value="replace" checked> Ganti Ulang (Replace)</label>
        <label style="cursor:pointer;"><input type="radio" name="recMode" value="append"> Tambah (Append)</label>
      </div>
      <button id="btn-stop-cam" style="background:#c0392b; color:#fff; border:none; padding:4px 10px; border-radius:4px; font-size:11px; cursor:pointer; font-weight:bold;">Matikan Kamera</button>
    </div>

    <div style="display: flex; gap: 16px; margin-bottom: 14px;">
      <div style="position: relative; width: 280px; height: 210px; background: #000; border-radius: 8px; overflow: hidden; flex-shrink: 0; display:flex; align-items:center; justify-content:center;">
        <video id="dash-video" style="width: 100%; height: 100%; object-fit: cover; transform: scaleX(-1); display:none;" autoplay muted playsinline></video>
        <div id="dash-placeholder" style="color: #888; font-size: 12px; text-align: center; padding: 10px;">Kamera standby.<br>Klik tombol gesture untuk mulai.</div>
        <div id="dash-countdown" style="display:none; position: absolute; font-size: 54px; font-weight: bold; color: #f1c40f; text-shadow: 0 2px 8px rgba(0,0,0,0.8);">3</div>
        <div id="dash-rec-bar" style="display:none; position: absolute; bottom: 8px; left: 8px; right: 8px; background: rgba(231,76,60,0.9); color:#fff; font-size:11px; padding: 4px; text-align:center; border-radius: 4px; font-weight:bold;">Merekam...</div>
      </div>

      <div style="flex-grow: 1; display: grid; grid-template-columns: 1fr 1fr; gap: 6px; align-content: start;">
        <div class="stat-badge" id="b-POINT" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">POINT: <b>0</b></div>
        <div class="stat-badge" id="b-PEACE" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">PEACE: <b>0</b></div>
        <div class="stat-badge" id="b-THREE_FINGERS" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">THREE: <b>0</b></div>
        <div class="stat-badge" id="b-FOUR_FINGERS" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">FOUR: <b>0</b></div>
        <div class="stat-badge" id="b-OK_SIGN" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">OK_SIGN: <b>0</b></div>
        <div class="stat-badge" id="b-OPEN_PALM" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">PALM: <b>0</b></div>
        <div class="stat-badge" id="b-THUMBS_UP" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">THUMBS: <b>0</b></div>
        <div class="stat-badge" id="b-FIST" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">FIST: <b>0</b></div>
      </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 12px;">
      <button class="rec-btn" id="btn-POINT">POINT</button>
      <button class="rec-btn" id="btn-PEACE">PEACE</button>
      <button class="rec-btn" id="btn-THREE_FINGERS">THREE</button>
      <button class="rec-btn" id="btn-FOUR_FINGERS">FOUR</button>
      <button class="rec-btn" id="btn-OK_SIGN">OK_SIGN</button>
      <button class="rec-btn" id="btn-OPEN_PALM">PALM</button>
      <button class="rec-btn" id="btn-THUMBS_UP">THUMBS</button>
      <button class="rec-btn" id="btn-FIST">FIST</button>
    </div>

    <div style="display:flex; justify-content:space-between; align-items:center; border-top:1px solid #333; padding-top:10px;">
      <div style="display:flex; gap:8px;">
        <button id="btn-synth" style="background:#34495e; color:#ecf0f1; border:none; padding:6px 12px; border-radius:5px; font-size:11px; cursor:pointer;">Muat Baseline Sintetis</button>
        <button id="btn-reset" style="background:#7f8c8d; color:#ecf0f1; border:none; padding:6px 12px; border-radius:5px; font-size:11px; cursor:pointer;">Reset Data</button>
      </div>
      <div id="dash-info" style="font-size:12px; color:#2ecc71;">Siap merekam data.</div>
    </div>
  `;

  document.querySelector('#output-body').appendChild(container);

  const style = document.createElement('style');
  style.textContent = `
    .rec-btn { background: #27ae60; color: white; border: none; padding: 8px 4px; border-radius: 6px; font-size: 11px; font-weight: bold; cursor: pointer; transition: 0.2s; }
    .rec-btn:hover { background: #2ecc71; }
    .rec-btn:disabled { background: #555; cursor: not-allowed; }
  `;
  container.appendChild(style);

  let isRecording = false;

  function stopCamera() {
    if (window.activeDatasetStream) {
      window.activeDatasetStream.getTracks().forEach(t => t.stop());
      window.activeDatasetStream = null;
    }
    const video = document.getElementById('dash-video');
    const placeholder = document.getElementById('dash-placeholder');
    if (video) video.style.display = 'none';
    if (placeholder) {
      placeholder.style.display = 'block';
      placeholder.innerHTML = 'Kamera dinonaktifkan.<br>Klik tombol gesture untuk mengaktifkan kembali.';
    }
    const info = document.getElementById('dash-info');
    if (info) { info.innerText = 'Kamera dimatikan.'; info.style.color = '#e74c3c'; }
  }

  function updateBadgeCounts(counts, total) {
    document.getElementById('dash-total-badge').innerText = `Total: ${total} Sampel`;
    for (const [key, val] of Object.entries(counts)) {
      const el = document.getElementById(`b-${key}`);
      if (el) {
        const icon = el.innerText.split(' ')[0];
        el.innerHTML = `${icon} ${key.split('_')[0]}: <b>${val}</b>`;
        if (val >= 10) el.style.background = '#1e8449';
        else if (val > 0) el.style.background = '#d35400';
        else el.style.background = '#2d3436';
      }
    }
  }

  async function getCameraStream() {
    if (window.activeDatasetStream && window.activeDatasetStream.active) {
      return window.activeDatasetStream;
    }
    const stream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 480 }, height: { ideal: 360 }, facingMode: "user" }
    });
    window.activeDatasetStream = stream;
    return stream;
  }

  async function triggerBurst(label) {
    if (isRecording) return;
    isRecording = true;
    container.querySelectorAll('.rec-btn').forEach(b => b.disabled = true);

    const video = document.getElementById('dash-video');
    const placeholder = document.getElementById('dash-placeholder');
    const cd = document.getElementById('dash-countdown');
    const recBar = document.getElementById('dash-rec-bar');
    const info = document.getElementById('dash-info');

    const mode = document.querySelector('input[name="recMode"]:checked')?.value || 'replace';

    placeholder.style.display = 'none';
    video.style.display = 'block';
    info.innerText = `Menyiapkan kamera untuk [${label}]...`;
    info.style.color = '#f1c40f';

    try {
      const stream = await getCameraStream();
      video.srcObject = stream;
      await video.play();

      cd.style.display = 'block';
      for (let c = 3; c > 0; c--) {
        cd.innerText = c;
        await new Promise(r => setTimeout(r, 1000));
      }
      cd.style.display = 'none';

      recBar.style.display = 'block';
      info.innerText = `Merekam [${label}]! Miringkan/gerakkan tangan sedikit...`;
      info.style.color = '#e74c3c';

      const canvas = document.createElement('canvas');
      canvas.width = 400;
      canvas.height = 300;
      const ctx = canvas.getContext('2d');
      const frames = [];

      for (let i = 0; i < 10; i++) {
        ctx.drawImage(video, 0, 0, 400, 300);
        frames.push(canvas.toDataURL('image/jpeg', 0.65));
        recBar.innerText = `Merekam Frame ${i+1}/10...`;
        await new Promise(r => setTimeout(r, 200));
      }

      recBar.style.display = 'none';
      info.innerText = `Mengekstrak MediaPipe untuk [${label}]...`;
      info.style.color = '#3498db';

      const res = await google.colab.kernel.invokeFunction('notebook.ProcessBurstFrames', [label, frames, mode], {});
      
      let data = null;
      if (res && res.data) {
        if (res.data['application/json']) {
          data = res.data['application/json'];
        } else if (res.data['text/plain']) {
          try {
            data = JSON.parse(res.data['text/plain']);
          } catch(e) {
            try { data = JSON.parse(res.data['text/plain'].replace(/'/g, '"')); } catch(e2){}
          }
        }
      }

      if (data && data.counts) {
        updateBadgeCounts(data.counts, data.total);
        const actionWord = mode === 'replace' ? 'Ganti ulang' : 'Tambah';
        info.innerText = `[${label}]: ${actionWord} berhasil! (+${data.valid} frame valid).`;
        info.style.color = '#2ecc71';
      } else {
        info.innerText = `Berhasil diproses (data tersimpan di Python).`;
        info.style.color = '#f1c40f';
      }

    } catch (err) {
      info.innerText = `Error Kamera: ${err.message}`;
      info.style.color = '#e74c3c';
    } finally {
      isRecording = false;
      container.querySelectorAll('.rec-btn').forEach(b => b.disabled = false);
    }
  }

  // Bind Buttons
  const gestures = ["POINT", "PEACE", "THREE_FINGERS", "FOUR_FINGERS", "OK_SIGN", "OPEN_PALM", "THUMBS_UP", "FIST"];
  gestures.forEach(g => {
    const btn = document.getElementById(`btn-${g}`);
    if (btn) btn.onclick = () => triggerBurst(g);
  });

  document.getElementById('btn-stop-cam').onclick = stopCamera;

  document.getElementById('btn-synth').onclick = async () => {
    const info = document.getElementById('dash-info');
    info.innerText = "Memuat baseline...";
    const res = await google.colab.kernel.invokeFunction('notebook.LoadSyntheticBaseline', [], {});
    let data = res.data['application/json'] || JSON.parse(res.data['text/plain']?.replace(/'/g, '"') || '{}');
    if (data.counts) updateBadgeCounts(data.counts, data.total);
    info.innerText = `Baseline aktif (${data.total || 320} sampel)!`;
    info.style.color = '#2ecc71';
  };

  document.getElementById('btn-reset').onclick = async () => {
    const info = document.getElementById('dash-info');
    const res = await google.colab.kernel.invokeFunction('notebook.ResetDataset', [], {});
    let data = res.data['application/json'] || JSON.parse(res.data['text/plain']?.replace(/'/g, '"') || '{}');
    if (data.counts) updateBadgeCounts(data.counts, 0);
    info.innerText = "Dataset telah dikosongkan.";
    info.style.color = '#e67e22';
  };
}
initDatasetStudio();
"""

display(Javascript(DASHBOARD_JS))


## 1.3 Pelatihan Model Random Forest dan Evaluasi Metrik

### Mengapa Memilih Random Forest untuk Klasifikasi Landmark?
* **Inferensi Cepat (Sub-milidetik):** Random Forest pada 42 fitur hanya membutuhkan waktu komputasi kurang dari 1 milidetik di CPU, sehingga sangat ideal untuk loop kendali 30 FPS tanpa memerlukan GPU Colab.
* **Non-linear Decision Boundaries:** Mampu memisahkan kombinasi fitur sendi jari yang kompleks dengan membentuk himpunan pohon keputusan (*ensemble of decision trees*).
* **Tahan Terhadap Overfitting:** Teknik agregasi bootstrap (*bagging*) mengurangi variansi prediksi secara efektif.

---

### Instruksi Tugas Mahasiswa:
Tentukan hyperparameter pada sel di bawah ini, lalu jalankan pelatihan. Sel akan menampilkan:
1. Skor Akurasi pada data uji (Test Set 20%).
2. Laporan performa rinci (Precision, Recall, F1-Score).
3. Matriks Kebingungan (Confusion Matrix Heatmap).


In [ ]:
# --- TENTUKAN HYPERPARAMETER DI SINI ---
n_trees = 100
max_tree_depth = 12
test_split_ratio = 0.20
random_seed = 42
# ---------------------------------------

import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

if len(DATASET_Y) < 16:
    print("[PERINGATAN] Dataset masih kosong atau terlalu sedikit!")
    print("Silakan rekam sampel di sel 1.2 atau klik tombol 'Muat Baseline Sintetis'.")
else:
    X = np.array(DATASET_X)
    y = np.array(DATASET_Y)

    # Pembagian data Train dan Test secara terstratifikasi
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_split_ratio, random_state=random_seed, stratify=y
    )

    print(f"Melatih Random Forest pada {len(X_train)} sampel pelatihan...")
    rf_model = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=max_tree_depth,
        random_state=random_seed
    )
    rf_model.fit(X_train, y_train)

    # Evaluasi pada Data Uji
    y_pred = rf_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Akurasi Model pada Data Uji: {acc * 100:.2f}%\n")

    print("Laporan Performa Klasifikasi:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Visualisasi Matriks Kebingungan (Confusion Matrix)
    cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_model.classes_)
    disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
    plt.title(f"Confusion Matrix Gesture Classifier (Akurasi: {acc*100:.1f}%)")
    plt.tight_layout()
    plt.show()

    # Simpan model untuk digunakan pada Bab 3
    joblib.dump(rf_model, "gesture_rf_model.joblib")
    print("Model disimpan ke 'gesture_rf_model.joblib'.")


### Memahami Hasil Evaluasi Model

#### 1. Mengapa Nilai Akurasi Sering Mencapai 100% pada Data Uji?
Jika Anda merekam data dalam burst 10 frame berurutan saat tangan menahan satu pose, frame uji (2 sampel) memiliki kemiripan geometris yang sangat tinggi dengan frame latih (8 sampel) dalam rentang waktu yang sama (*temporal autocorrelation*). 

Geometri 42-fitur antar-gesture juga sangat kontras (misalnya jarak sendi jari pada `FIST` dan `OPEN_PALM` sangat berbeda jauh), sehingga Random Forest dapat memisahkannya dengan mudah.

#### 2. Kapan Ujian Sebenarnya Terjadi?
Ujian kemampuan generalisasi model yang sebenarnya bukan pada data uji statis di atas, melainkan saat tangan digerakkan secara dinamis pada sudut dan jarak yang baru. Hal ini akan kita uji langsung di sel berikutnya.


## 1.4 Pengujian Model Real-Time Mandiri (Offline / Tanpa Robot)

Sebelum menghubungkan model ke robot fisik di lab, kita perlu memastikan model dapat mengenali pose tangan secara lancar saat Anda menggerakkan tangan secara bebas di depan webcam.

Jalankan sel berikut untuk memulai sesi uji coba offline selama 30 detik. Tampilkan berbagai variasi pose ke kamera dan perhatikan teks prediksi serta persentase keyakinan (*confidence*) yang muncul pada layar.


In [ ]:
#@title 1.4 Pengujian Kamera Real-Time Mandiri { display-mode: "form" }
# Harap jangan mengubah isi sel ini
# 1.4 Fungsi Prediksi & Pengujian Kamera Real-Time (Offline / Tanpa UR5)
import time
import asyncio
import cv2
import numpy as np
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js

def predict_gesture_with_confidence(feature_vector: np.ndarray):
    """Menerima 42-fitur normalisasi dan mengembalikan nama gesture + probabilitas."""
    feats = feature_vector.reshape(1, -1)
    probs = rf_model.predict_proba(feats)[0]
    best_idx = np.argmax(probs)
    pred_label = rf_model.classes_[best_idx]
    confidence = float(probs[best_idx])
    return pred_label, confidence


OFFLINE_TESTER_JS = """
async function initOfflineTester() {
  const existing = document.getElementById('offline-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'offline-box';
  container.style.fontFamily = 'monospace';
  container.style.background = '#1a1a1a';
  container.style.color = '#fff';
  container.style.padding = '12px';
  container.style.borderRadius = '8px';
  container.style.display = 'inline-block';

  const header = document.createElement('div');
  header.innerHTML = '<b>Uji Coba Model Gesture Real-Time (Offline / Tanpa Robot)</b>';
  header.style.marginBottom = '6px';
  container.appendChild(header);

  const video = document.createElement('video');
  video.style.borderRadius = '6px';
  video.style.transform = 'scaleX(-1)';
  video.width = 400;
  video.height = 300;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  container.appendChild(video);

  const hud = document.createElement('div');
  hud.id = 'offline-hud';
  hud.style.marginTop = '6px';
  hud.style.fontSize = '14px';
  hud.innerHTML = 'Status: Membuka kamera...';
  container.appendChild(hud);

  const btn = document.createElement('button');
  btn.innerText = 'Hentikan Uji Coba';
  btn.style.marginTop = '8px';
  btn.style.background = '#e74c3c';
  btn.style.color = '#fff';
  btn.style.border = 'none';
  btn.style.padding = '6px 12px';
  btn.style.borderRadius = '4px';
  btn.style.cursor = 'pointer';

  btn.onclick = () => {
    window.offlineActive = false;
    if (window.offlineStream) window.offlineStream.getTracks().forEach(t => t.stop());
    btn.innerText = '⏹️ Pengujian Selesai';
    btn.disabled = true;
    hud.innerHTML = 'Pengujian dinonaktifkan.';
  };
  container.appendChild(btn);

  document.querySelector('#output-body').appendChild(container);

  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 400, height: 300}});
  window.offlineStream = stream;
  video.srcObject = stream;
  await video.play();

  const canvas = document.createElement('canvas');
  canvas.width = 400;
  canvas.height = 300;
  const ctx = canvas.getContext('2d');

  window.offlineActive = true;
  window.captureOfflineFrame = function() {
    if (!window.offlineActive) return null;
    ctx.drawImage(video, 0, 0, 400, 300);
    return canvas.toDataURL('image/jpeg', 0.6);
  };

  window.updateOfflineHUD = function(label, conf) {
    const el = document.getElementById('offline-hud');
    if (!el || !window.offlineActive) return;
    const pct = Math.round(conf * 100);
    const color = conf >= 0.7 ? '#2ecc71' : '#e67e22';
    el.innerHTML = `Prediksi: <b style="color:${color}; font-size:16px;">[${label}]</b> | Confidence: <b>${pct}%</b>`;
  };
}
initOfflineTester();
"""

async def test_gesture_model_offline(duration_seconds: float = 30.0):
    """Menjalankan visualisasi inferensi real-time di browser tanpa koneksi ke robot UR5."""
    display(Javascript(OFFLINE_TESTER_JS))
    await asyncio.sleep(2.0)

    print("Pengujian kamera offline aktif! Tunjukkan pose tangan ke kamera.")
    print(f"Pengujian akan berjalan otomatis selama {duration_seconds} detik (atau klik tombol Hentikan).\\n")

    t_start = time.time()
    try:
        while (time.time() - t_start) < duration_seconds:
            is_active = eval_js('window.offlineActive ? true : false')
            if not is_active:
                break

            frame_b64 = eval_js('window.captureOfflineFrame ? window.captureOfflineFrame() : null')
            if frame_b64 is None:
                await asyncio.sleep(0.05)
                continue

            header, encoded = frame_b64.split(',', 1) if ',' in frame_b64 else ('', frame_b64)
            img_bytes = base64.b64decode(encoded)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
            rgb = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)

            lms = hands_detector.detect_landmarks(rgb)
            if lms is not None:
                feats = extract_normalized_features(lms)
                if feats is not None:
                    label, conf = predict_gesture_with_confidence(feats)
                    eval_js(f'window.updateOfflineHUD ? window.updateOfflineHUD("{label}", {conf}) : null')
            else:
                eval_js('window.updateOfflineHUD ? window.updateOfflineHUD("TIDAK ADA TANGAN", 0.0) : null')

            await asyncio.sleep(0.04)

    finally:
        eval_js('if (window.offlineStream) { window.offlineStream.getTracks().forEach(t => t.stop()); window.offlineActive = false; }')
        print("Pengujian offline selesai.")


# Jalankan pengujian offline kamera (30 detik)
await test_gesture_model_offline(duration_seconds=30.0)


# Bab 2: Perintah Kompleks Lengan Robot UR5 Berbasis LLM (Large Language Model)

## 2.0 Konteks Konseptual: Dari Bahasa Manusia ke Aksi Fisik

Pada Bab 1, mahasiswa membangun **persepsi**: kamera → landmark tangan → 42 fitur → label gesture. Pada Bab 3, label gesture tersebut diterjemahkan langsung menjadi perintah waypoint. Bab 2 mengisi kekosongan di antaranya: bagaimana **kalimat bahasa alami** diubah menjadi rencana aksi yang bisa dieksekusi robot.

### Masalah pada Perintah Teks Biasa
Jika kita meneruskan kalimat manusia apa adanya ke aktuator robot:
* Kalimat bersifat **ambig** — "taruh di sebelah kotak" tidak memiliki koordinat yang bisa dieksekusi.
* Tidak ada jaminan **urutan** — robot bisa menjepit sebelum turun ke meja.
* Tidak ada **konteks keadaan** — model tidak tahu apakah gripper sedang membawa barang atau posisi tujuan sudah penuh.

### Solusi: LLM sebagai Perencana, Bukan Pengendali Langsung
LLM hanya boleh menghasilkan **simbol aksi** dari daftar tertutup. Eksekusi fisik tetap dilakukan oleh `robot_server.py`, yang memegang guardrail, batas workspace, dan kunci sewa (*lease*). Pemisahan ini membuat robot tetap aman meskipun model salah bicara.

| Lapisan | Tanggung jawab | Lokasi di modul |
| --- | --- | --- |
| Persepsi | Gestur / kalimat → niat | Bab 1 & 2.1 |
| Perencanaan | Niat → satu aksi valid | Bab 2.2–2.3 |
| Validasi | Guardrail & aturan okupansi | Bab 2.3 (simulasi) & server |
| Aktuasi | Aksi → gerakan UR5 fisik | Bab 2.4 |

> **Aturan penting:** model yang sedang berpikir (*thinking mode*) menolak `tool_choice` yang dipaksa. Jangan gunakan `tool_choice={"type": "function", ...}`. Kirim saja pesan system/user biasa.

> **Kosa kata aksi server (sumber kebenaran):** `move to home`, `box`, `pos1`–`pos4`, `pos1-up`–`pos4-up`, `open gripper`, `close gripper`, `task_completed`, `reject_task`.


## 2.1 Chat ke LLM

### Mengapa Memulai dari Chat Biasa?
Sebelum memerintahkan robot, kita memastikan dulu bahwa koneksi ke API LLM bekerja. Chat completion adalah kontrak paling sederhana: kita mengirim **daftar pesan** dengan peran `system` (aturan) dan `user` (instruksi), lalu menerima satu balasan `assistant`.

Pada sel ini belum ada tool, belum ada robot, dan belum ada WebSocket. Tujuannya hanya membuktikan bahwa API key, base URL, dan model dapat diakses dari lingkungan Colab.

> **Spesifikasi penting:** gunakan `model="deepseek-flash"` dan jangan mengirim parameter `tool_choice`. Beberapa model reasoning mengembalikan HTTP 400 bila tool dipaksa.

Kunci API diambil dari **Colab Secrets** dengan nama `DEEPSEEK_API_KEY`. Jangan pernah menempelkan kunci langsung di cell.


### Apa yang Kita Pelajari dari Output Ini?
Perhatikan bahwa `llm_history` bertambah dua entri setiap kali `llm_chat()` dipanggil: satu pesan `user` dan satu pesan `assistant`. Riwayat inilah yang membuat percakapan terasa "berkelanjutan" — LLM melihat seluruh percakapan sebelumnya pada setiap permintaan.

Coba kirim dua prompt berurutan dan amati bagaimana jawaban kedua merujuk pada jawaban pertama. Pada Bab 2.3 nanti, konsep riwayat ini berubah menjadi **action history**, bukan sekadar obrolan.


In [ ]:
!pip install -q openai

import asyncio
import json
import os
import re
import time
from openai import OpenAI

try:
    from google.colab import userdata as _userdata
except ImportError:
    _userdata = None

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL_NAME = "deepseek-flash"
llm_history = []

def get_deepseek_key() -> str:
    key = ""
    if _userdata:
        try:
            key = (_userdata.get("DEEPSEEK_API_KEY") or "").strip()
        except Exception:
            pass
    return key or os.environ.get("DEEPSEEK_API_KEY", "").strip()

def llm_chat(user_text: str) -> str:
    key = get_deepseek_key()
    assert key, "Tambahkan DEEPSEEK_API_KEY di Colab Secrets (ikon kunci di sidebar)."
    llm_history.append({"role": "user", "content": user_text})
    client = OpenAI(api_key=key, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(model=MODEL_NAME, messages=llm_history)
    answer = resp.choices[0].message.content or ""
    llm_history.append({"role": "assistant", "content": answer})
    return answer

print(llm_chat("Jelaskan dalam satu kalimat: apa bedanya obrolan dan perintah robot?"))
assert llm_history and llm_history[-1]["role"] == "assistant"
print("Total pesan dalam riwayat:", len(llm_history))


## 2.2 Konteks Prompt Engineering

### Masalah: LLM Tidak Tahu Keadaan Meja
Model bahasa hanya melihat teks yang kita kirim. Ia tidak bisa melihat kamera, tidak tahu item sudah dipindahkan mahasiswa sebelumnya, dan tidak tahu apakah gripper sedang kosong. Tanpa informasi itu, LLM akan **mengarang** kondisi dunia.

### Solusi: Prompt sebagai Snapshot Dunia
Kita menyuntikkan keadaan terkini ke dalam system prompt setiap kali meminta keputusan:

| Field konteks | Arti | Kenapa wajib ada |
| --- | --- | --- |
| `current_item_positions` | Letak tiap item (`box`, `pos1`–`pos4`, `gripper`) | Satu-satunya *ground truth*; mencegah LLM mengambil item yang sudah dipindah |
| `holding_item` | Item yang sedang digenggam | Mencegah gripper mengambil dua barang |
| `waypoint` | Posisi/ ketinggian lengan saat ini | Mencegah menjepit di udara (`-up`) |
| `history` | Daftar aksi yang sudah dijalankan | Mencegah LLM mengulang siklus pick-and-place yang sudah selesai |

Dua aturan yang sering tertukar:

* **Reject** hanya boleh dipanggil di awal, misalnya item sudah ada di tujuan atau posisi tujuan penuh.
* **Task completed** dipanggil setelah barang berhasil diletakkan — bukan `reject_task`.

> Mahasiswa dapat mengubah `context` pada sel berikut, lalu mengamati bagaimana perubahan satu field mengubah jawaban LLM.


In [ ]:
ALLOWED_ACTIONS = [
    "move to home", "box", "pos1", "pos2", "pos3", "pos4",
    "pos1-up", "pos2-up", "pos3-up", "pos4-up",
    "open gripper", "close gripper", "task_completed", "reject_task",
]

def build_system_prompt(ctx: dict) -> str:
    items = ", ".join(f"{k}={v}" for k, v in ctx["current_item_positions"].items())
    history = " | ".join(ctx.get("history", [])) or "Belum ada langkah."
    return "\n".join([
        "Kamu adalah perencana aksi untuk lengan UR5 dengan gripper Robotiq.",
        "Jawab dengan SATU aksi saja, diambil tepat dari daftar ini: " + ", ".join(ALLOWED_ACTIONS) + ".",
        "Posisi item saat ini (ground truth): " + items + ".",
        "Sedang membawa: " + str(ctx.get("holding_item") or "tidak ada") + ".",
        "Waypoint saat ini: " + str(ctx.get("waypoint") or "home") + ".",
        "Riwayat aksi: " + history + ".",
        "Aturan: pos1..pos4 dan box adalah posisi meja; posN-up adalah posisi aman untuk berpindah, dilarang menjepit di sana.",
        "Jika item sudah berada di tujuan atau tujuan penuh, jawab reject_task.",
        "Jika semua instruksi selesai, jawab task_completed. Jangan menjawab reject_task setelah barang diletakkan.",
    ])

context = {
    "current_item_positions": {"item_1": "box", "item_2": "pos3"},
    "holding_item": None,
    "waypoint": None,
    "history": [],
}

prompt = build_system_prompt(context)
assert all(a in prompt for a in ALLOWED_ACTIONS)
assert "item_1=box" in prompt and "Riwayat aksi" in prompt
print(prompt)

def llm_plan(prompt_task: str, ctx: dict) -> str:
    key = get_deepseek_key()
    client = OpenAI(api_key=key, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": build_system_prompt(ctx)},
            {"role": "user", "content": prompt_task},
        ],
    )
    return resp.choices[0].message.content or ""

print("\nContoh jawaban LLM:", llm_plan("Ambil item 1 lalu taruh di pos 2", context))


## 2.3 Simulasi Action Planner

### Mengapa Tidak Langsung ke Robot?
Satu kesalahan urutan pada robot fisik bisa merusak gripper atau meja. Karena itu, logika perencanaan diuji lebih dulu pada **simulator deterministik** yang memakai aturan sama dengan server: grip hanya di meja, satu item per posisi, satu item di gripper.

### Perbedaan Batch Planning dan Reactive Closed-Loop
Pada *batch planning*, LLM diminta sekali menghasilkan seluruh daftar aksi sekaligus. Jika satu aksi ditolak guardrail, seluruh rencana menjadi invalid. Pada **reactive closed-loop**, LLM memutuskan **satu aksi per putaran**:

1. Kirim konteks + riwayat aksi → terima satu aksi.
2. Simulator memeriksa aksi. Bila ditolak, **alasan penolakan dikembalikan ke LLM** sebagai feedback.
3. LLM memperbaiki diri (maksimal 3 kali percobaan) atau mengubah strategi.
4. Ulangi sampai `task_completed` atau `reject_task`.

Pendekatan ini meniru apa yang dilakukan `robot_server.py` secara internal, hanya saja tanpa menggerakkan hardware.

### Yang Harus Kamu Amati
Perhatikan baris `Step n: ... DITOLAK (...)` pada keluaran. Itu adalah momen *self-correction*: LLM belajar dari guardrail tanpa campur tangan manusia.


In [ ]:
class ActionSimulator:
    TABLE = ("box", "pos1", "pos2", "pos3", "pos4")

    def __init__(self, positions: dict):
        self.positions = dict(positions)
        self.holding = None
        self.waypoint = None

    def snapshot(self):
        return {"current_item_positions": dict(self.positions),
                "holding_item": self.holding, "waypoint": self.waypoint}

    def apply(self, action: str):
        action = action.strip()
        if action in ("task_completed", "reject_task"):
            return True, action, self.snapshot()
        if action == "move to home":
            self.waypoint = None
            return True, action, self.snapshot()
        if action in self.TABLE or action.endswith("-up"):
            self.waypoint = action
            return True, action, self.snapshot()
        if action in ("open gripper", "close gripper"):
            if self.waypoint not in self.TABLE:
                return False, "Grip hanya diperbolehkan di posisi meja, bukan di waypoint -up.", self.snapshot()
            if action == "close gripper":
                if self.holding:
                    return False, "Gripper sudah membawa item.", self.snapshot()
                found = next((k for k, v in self.positions.items() if v == self.waypoint), None)
                if not found:
                    return False, "Tidak ada item di posisi ini.", self.snapshot()
                self.positions[found] = "gripper"
                self.holding = found
            else:
                if not self.holding:
                    return False, "Gripper kosong, tidak ada yang dilepas.", self.snapshot()
                if self.waypoint in self.positions.values():
                    return False, "Posisi tujuan sudah penuh.", self.snapshot()
                self.positions[self.holding] = self.waypoint
                self.holding = None
            return True, action, self.snapshot()
        return False, "Aksi tidak dikenal: " + action, self.snapshot()

# Uji aturan simulator tanpa LLM dan tanpa robot.
sim = ActionSimulator({"item_1": "box", "item_2": "pos3"})
ok, reason, _ = sim.apply("close gripper")
assert not ok, "Grip di udara harus ditolak"
ok, reason, _ = sim.apply("box")
assert ok
ok, reason, _ = sim.apply("close gripper")
assert ok and sim.holding == "item_1"
ok, reason, _ = sim.apply("pos2-up")
assert ok
ok, reason, _ = sim.apply("open gripper")
assert not ok, "Grip di -up harus ditolak"
ok, reason, _ = sim.apply("pos2")
ok, reason, _ = sim.apply("open gripper")
assert ok and sim.positions["item_1"] == "pos2"
print("Semua aturan simulator lolos. State akhir:", sim.snapshot())

def parse_action(text: str):
    for a in ALLOWED_ACTIONS:
        if re.search(r"\b" + re.escape(a) + r"\b", text, re.IGNORECASE):
            return a
    return None

def plan_and_simulate(task: str, sim: ActionSimulator, max_steps: int = 12):
    ctx = sim.snapshot()
    history = []
    for step in range(max_steps):
        raw = llm_plan(task, ctx)
        act = parse_action(raw)
        if not act:
            history.append("Step %d: jawaban tidak valid (%s)" % (step + 1, raw[:60]))
            ctx["feedback"] = "Pilih tepat satu aksi dari daftar yang diizinkan."
            continue
        ok, reason, snap = sim.apply(act)
        if not ok:
            history.append("Step %d: %s DITOLAK (%s)" % (step + 1, act, reason))
            ctx["feedback"] = reason
            ctx["history"] = history
            continue
        history.append("Step %d: %s" % (step + 1, act))
        ctx = snap
        ctx["history"] = history
        if act in ("task_completed", "reject_task"):
            break
    return sim, history

sim2 = ActionSimulator({"item_1": "box"})
sim2, riwayat = plan_and_simulate("Ambil item 1 lalu taruh di pos 1", sim2)
print("\n".join(riwayat))
print("State akhir:", sim2.snapshot())


## 2.4 Implementasi Langsung ke Robot UR5

### Arsitektur Pesan: Satu Koneksi, Satu Siklus Hidup
Setelah klik **Send Task**, klien membuka satu WebSocket dan melakukan tiga langkah di koneksi yang **sama**:

```
HANDSHAKE  →  PROMPT_TASK  →  PLAN_ACK / STEP_PROGRESS / PLAN_COMPLETE
```

Ini penting karena server mengirim seluruh balasan melalui objek `websocket` yang menerima task. Jika klien menutup koneksi lebih awal, server gagal mengirim balasan dan task berakhir `FAILED`. Di sel di bawah, koneksi baru ditutup setelah pesan akhir diterima.

### Safety Gates yang Masih Dipegang Server
Meskipun UI hanya punya tiga tombol, server tetap melindungi hardware:

* **Aturan posisi tujuan:** jika item sudah berada di tujuan sejak awal, task ditolak tanpa menggerakkan lengan.
* **Okupansi:** satu posisi hanya menampung satu item; gripper hanya menampung satu item.
* **Redzone & ketinggian:** transisi lateral hanya di $z = 0$; kontak meja di $z = -8$; menjepit hanya di posisi meja, bukan di `-up`.

> **Contoh yang sering membingungkan:** task *"Pick item 1 and place it on pos 1"* bisa selesai dalam **0 langkah** bila `item_1` memang sudah di `pos1`. Itu bukan bug — sesuai aturan target position. Klik **Reset / Calibrate** dulu bila ingin melihat gerakan fisik.

Sel berikut adalah **infrastruktur**: jangan diubah. Cukup jalankan sekali, lalu isi prompt dan klik **Send Task** berulang kali. Tombol **E-STOP** mengirim `EMERGENCY_STOP` dan menunggu `EMERGENCY_STOP_ACK`.


In [ ]:
# CELL 9 — Kirim Task & E-STOP ke UR5 (tanpa live log, tanpa loop)
# Jalankan cell sekali untuk menampilkan UI. Tombol tetap bisa dipakai
# selama kernel Colab hidup.

import asyncio
import json
import os
import time
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import userdata as _userdata
except ImportError:
    _userdata = None


def _api_key() -> str:
    key = ""
    if _userdata:
        try:
            key = (_userdata.get("DEEPSEEK_API_KEY") or "").strip()
        except Exception:
            pass
    return key or os.environ.get("DEEPSEEK_API_KEY", "").strip()


def _ws_url() -> str:
    url = ""
    if _userdata:
        try:
            url = (_userdata.get("TUNNEL_WS_URL") or "").strip()
        except Exception:
            pass
    return url or os.environ.get("TUNNEL_WS_URL", "").strip() or "wss://ur5.fawzqi.my.id"


def _fire(coro):
    # Colab memanggil callback widget saat loop kernel sedang berjalan,
    # jadi jangan buat loop baru — jadwalkan di loop yang sudah ada.
    task = asyncio.ensure_future(coro)
    task.add_done_callback(lambda t: t.exception() if not t.cancelled() else None)


async def _send(payload: dict, wait_for: str):
    import websockets
    async with websockets.connect(_ws_url(), open_timeout=8) as ws:
        await ws.send(json.dumps({
            "msg_id": f"m_{int(time.time()*1000) % 10**6}",
            "timestamp": time.time(),
            **payload,
        }))
        # Jangan tutup koneksi sebelum server selesai: server mengirim
        # STEP_PROGRESS/PLAN_COMPLETE lewat koneksi ini. Kalau ditutup lebih
        # awal, task di server gagal saat mencoba mengirim balasan.
        while True:
            try:
                msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=300))
            except asyncio.TimeoutError:
                break
            mt = msg.get("msg_type")
            p = msg.get("payload", {})
            if mt == wait_for or mt == "ERROR" or (mt == "PLAN_ACK" and p.get("accepted") is False):
                break


def _on_send(_):
    prompt = task_box.value.strip()
    if not prompt or not _api_key():
        return
    session = globals().get("session")
    token = getattr(session, "token", None) if session else None
    _fire(_send({
        "msg_type": "PROMPT_TASK",
        "payload": {"prompt": prompt, "api_key": _api_key(), "model": "deepseek-flash", "token": token},
    }, "PLAN_COMPLETE"))


def _on_estop(_):
    _fire(_send({
        "msg_type": "EMERGENCY_STOP",
        "payload": {"reason": "E-STOP dari Colab"},
    }, "EMERGENCY_STOP_ACK"))


task_box = widgets.Text(value="Pick item 1 and place it on pos 1", description="Task:",
                        layout=widgets.Layout(width="430px"))
send_btn = widgets.Button(description="Send Task", button_style="success", icon="paper-plane",
                          layout=widgets.Layout(width="130px"))
estop_btn = widgets.Button(description="E-STOP", button_style="danger", icon="stop",
                           layout=widgets.Layout(width="110px"))

send_btn.on_click(_on_send)
task_box.on_submit(_on_send)
estop_btn.on_click(_on_estop)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>🤖 UR5 Task Control</h3>"),
    widgets.HBox([task_box, send_btn, estop_btn], layout=widgets.Layout(margin="6px 0")),
]))


### Rangkuman Bab 2

| Konsep | Fungsi di notebook | Hasil yang diamati |
| --- | --- | --- |
| Chat completion | `llm_chat()` | Riwayat pesan `user`/`assistant` |
| Prompt berkonteks | `build_system_prompt()` | System prompt memuat posisi item & riwayat |
| Perencanaan reaktif | `plan_and_simulate()` | Aksi per putaran + self-correction |
| Simulasi aman | `ActionSimulator.apply()` | Penolakan grip di udara / posisi penuh |
| Eksekusi fisik | Cell 2.4 | `PROMPT_TASK` → `PLAN_COMPLETE` di server |

### Checklist sebelum praktik

1. Simulator di 2.3 lolos semua `assert`.
2. Log `robot_server.py` menampilkan `Received message: [PROMPT_TASK]` setelah tombol ditekan.
3. Jika robot sedang dipakai kelompok lain, minta `release_lock()` atau tunggu habisnya sewa (maksimal 60 detik).
4. Uji tombol **E-STOP** sekali sebelum menjalankan task panjang.
5. Pahami bedanya `REJECTED` (ditolak di awal) dan `COMPLETED` (barang berhasil diletakkan).


# Bab 3: Kendali Teleoperasi Waypoint Lengan Robot Manipulator UR5

Pada Bab 3, kita mengintegrasikan model Machine Learning yang telah dilatih pada Bab 1 dengan sistem aktuasi lengan robot industri UR5 di Laboratorium B401 menggunakan kendali gestur tangan langsung.

## 3.1 Konfigurasi Kuadran Papan Catur dan Pemetaan Aksi

Area kerja meja robot dipetakan menggunakan kisi papan catur standar internasional (A1 hingga H8) dan sebuah wadah penyimpanan (`BOX`):
* Kuadran 1 (Q1) dipetakan ke petak `B2`
* Kuadran 2 (Q2) dipetakan ke petak `G2`
* Kuadran 3 (Q3) dipetakan ke petak `G7`
* Kuadran 4 (Q4) dipetakan ke petak `B7`
* Wadah Peletakan (`BIN`) dipetakan ke petak `BOX`

Fungsi `validate_quadrant_config` menjamin bahwa mahasiswa hanya memasukkan petak catur yang valid dan berada dalam batas jangkauan aman robot.


In [ ]:
# --- TULIS KONFIGURASI KUADRAN DI SINI ---
# 3.1 Konfigurasi Kuadran Chessboard & Pemetaan Aksi Gesture
# Mahasiswa dapat menyesuaikan petak target pada papan catur (A1 - H8) dan wadah BOX.
QUADRANT_CONFIG = {
    "Q1": "B2",   # Kuadran 1 (Depan-Kiri)
    "Q2": "G2",   # Kuadran 2 (Depan-Kanan)
    "Q3": "G7",   # Kuadran 3 (Belakang-Kanan)
    "Q4": "B7",   # Kuadran 4 (Belakang-Kiri)
    "BIN": "BOX", # Wadah peletakan barang
}

# Pemetaan dari hasil prediksi Machine Learning ke aksi robot
GESTURE_TO_ACTION = {
    "POINT": "HOVER_Q1",         # 1 Jari -> Hover Kuadran 1
    "PEACE": "HOVER_Q2",         # 2 Jari -> Hover Kuadran 2
    "THREE_FINGERS": "HOVER_Q3", # 3 Jari -> Hover Kuadran 3
    "FOUR_FINGERS": "HOVER_Q4",  # 4 Jari -> Hover Kuadran 4
    "OK_SIGN": "PICK_SEQUENCE",  # OK Sign -> Sekuens Pick (Hover -> Buka -> Turun -> Jepit -> Angkat)
    "OPEN_PALM": "RELEASE_BIN",  # Telapak Buka -> Sekuens Place (Ke BOX -> Turun -> Buka -> Angkat)
    "THUMBS_UP": "READY",        # Jempol -> Posisi netral / Home
    "FIST": None,                # Kepalan -> Hold / Netral (Safety Deadzone)
}

# ── Validasi Keabsahan Konfigurasi Kuadran ──
def validate_quadrant_config(config: dict):
    """Memastikan setiap petak catur valid sebelum mengirimkan rencana ke robot fisik."""
    valid_files = set("ABCDEFGH")
    valid_ranks = set("12345678")
    valid_squares = {f"{f}{r}" for f in valid_files for r in valid_ranks} | {"BOX"}

    required_keys = ["Q1", "Q2", "Q3", "Q4", "BIN"]
    for k in required_keys:
        if k not in config:
            raise KeyError(f"Kunci '{k}' tidak ditemukan pada QUADRANT_CONFIG! Harus mendefinisikan Q1, Q2, Q3, Q4, dan BIN.")
        
        sq = str(config[k]).strip().upper()
        if sq not in valid_squares:
            raise ValueError(
                f"Petak catur '{sq}' pada [{k}] TIDAK VALID!\\n"
                f"   Petak harus berada dalam rentang A1 - H8 (misal: 'B2', 'G7') atau 'BOX'."
            )

    print("Validasi Berhasil! Seluruh petak target valid:")
    for q_key, sq in config.items():
        print(f"   • {q_key:<4} -> Petak [{sq}]")


# Jalankan validasi
validate_quadrant_config(QUADRANT_CONFIG)


## 3.2 Filter Kestabilan Dwell Debouncer dan Idempotensi Gerak

### Mengapa Kita Tidak Mengirim Koordinat Tangan Kontinu Setiap Frame?
Dalam teleoperasi industri melalui internet, mengirimkan posisi tangan $(x, y, z)$ secara kontinu (misalnya 30 kali per detik) sangat berisiko. Adanya fluktuasi latensi jaringan (*jitter*) dapat menyebabkan lengan robot tersendat (*jerky*), melenceng dari batas aman, atau memicu *Protective Stop*.

Oleh karena itu, kita menggunakan **Perintah Makro Diskrit Berbobot Tinggi** yang dilindungi oleh **Dwell Debouncer**:
1. **Dwell Time (1.0 Detik):** Pengguna wajib menahan pose tangan secara stabil selama minimal 1.0 detik berturut-turut sebelum perintah dieksekusi. Gerakan tangan sekilas atau kedipan deteksi tidak akan memicu gerakan robot.
2. **Kunci Idempotensi (Latch-and-Hold):** Satu pose tangan yang ditahan lama hanya akan memicu 1 kali eksekusi gerak, mencegah robot mengulang-ulang aksi yang sama tanpa henti.
3. **Masa Pendinginan (Cooldown 1.0 Detik):** Setelah robot selesai bergerak, sistem mengunci pemicu selama 1 detik agar transisi tangan pengguna kembali ke posisi netral tidak dianggap sebagai perintah baru.


In [ ]:
#@title 3.2 Inisialisasi Filter Kestabilan Dwell Debouncer { display-mode: "form" }
# Harap jangan mengubah isi sel ini

import time

class TeleopDebouncer:
    """
    Filter kestabilan gesture:
      - Dwell Time: Tangan harus menahan gesture minimal 1.0 detik secara stabil.
      - Latch-and-Hold: Satu gesture hanya memicu 1 kali eksekusi gerak.
      - Cooldown: Jeda 1.0 detik setelah robot selesai bergerak untuk mencegah re-trigger.
    """
    def __init__(self, dwell_time_s: float = 1.0, cooldown_s: float = 1.0):
        self.dwell_time = dwell_time_s
        self.cooldown_time = cooldown_s
        self.current_candidate = None
        self.candidate_start_time = 0.0
        self.latched_gesture = None
        self.last_trigger_time = 0.0

    def update(self, detected_gesture: str) -> bool:
        now = time.time()
        if not detected_gesture or detected_gesture in ("FIST", "UNKNOWN", "NONE"):
            self.current_candidate = None
            self.latched_gesture = None
            return False

        if detected_gesture == self.latched_gesture:
            return False

        if (now - self.last_trigger_time) < self.cooldown_time:
            return False

        if detected_gesture != self.current_candidate:
            self.current_candidate = detected_gesture
            self.candidate_start_time = now
            return False

        elapsed = now - self.candidate_start_time
        if elapsed >= self.dwell_time:
            self.last_trigger_time = now
            self.latched_gesture = detected_gesture
            self.current_candidate = None
            return True

        return False

    def get_progress(self) -> float:
        if not self.current_candidate or self.candidate_start_time == 0.0:
            return 0.0
        return min(1.0, (time.time() - self.candidate_start_time) / self.dwell_time)

    def arm_cooldown(self):
        self.last_trigger_time = time.time()

debouncer = TeleopDebouncer(dwell_time_s=1.0, cooldown_s=1.0)
print("Filter Dwell Debouncer siap.")


## 3.3 Uji Coba Skenario Deterministik (Skenario Pick and Place)

Ini adalah jalur uji coba terprogram untuk memverifikasi seluruh rantai aksi:
1. `POINT` $\to$ Robot bergerak ke Kuadran 1 (`HOVER_Q1`).
2. `OK_SIGN` $\to$ Robot menjalankan makro mengambil benda (`PICK_AT_CURRENT`: turun, menjepit gripper, lalu mengangkat kembali).
3. `OPEN_PALM` $\to$ Robot membawa benda ke wadah penampung dan membukanya (`PLACE_BIN`).
4. `THUMBS_UP` $\to$ Robot kembali ke posisi istirahat (`HOME`).

Jalankan sel berikut untuk mengeksekusi skenario lengkap ke server lab.


In [ ]:
async def handle_gesture_trigger(gesture_name: str):
    """Menerima nama gesture terverifikasi, memetakan aksi, dan mengirimkannya ke robot."""
    action = GESTURE_TO_ACTION.get(gesture_name)
    if not action:
        return False

    print(f"\n[GESTURE] Terdeteksi: [{gesture_name}] -> Memicu Aksi: [{action}]")
    if not session.token:
        has_lock = await session.acquire_lock(duration_s=45.0)
        if not has_lock:
            return False

    success = await session.execute(action)
    debouncer.arm_cooldown()
    return success

print("Menjalankan simulasi skenario Pick and Place lengkap...")
scenario_gestures = ["POINT", "OK_SIGN", "OPEN_PALM", "THUMBS_UP"]

for g in scenario_gestures:
    await handle_gesture_trigger(g)
    await asyncio.sleep(1.0)

await session.release_lock()
print("Skenario selesai dijalankan.")


## 3.4 Teleoperasi Kamera Real-Time Interaktif (Ujian Praktik)

Pada tahap akhir ini, seluruh komponen diintegrasikan secara langsung:
`Kamera Laptop` $\to$ `MediaPipe` $\to$ `Random Forest` $\to$ `Dwell Debouncer` $\to$ `WebSocket Dispatcher` $\to$ `Lengan Robot UR5`.

### Petunjuk Penggunaan:
1. Jalankan sel kode di bawah ini. Widget video teleoperasi akan muncul di peramban.
2. Berikan izin akses kamera jika diminta.
3. Arahkan pose tangan ke kamera (`POINT`, `PEACE`, `OK_SIGN`, `OPEN_PALM`, `THUMBS_UP`).
4. **Tahan pose selama 1.0 detik** hingga bar Dwell mencapai 100%. Lengan robot di lab akan langsung bergerak mengeksekusi instruksi Anda.
5. Klik tombol **Hentikan Teleoperasi** jika ingin mengakhiri sesi lebih awal.


In [ ]:
#@title 3.4 Loop Teleoperasi Kamera Real-Time { display-mode: "form" }
# Harap jangan mengubah isi sel ini
# 3.4 Live Webcam Teleoperation Loop in Google Colab
import base64
import time
import asyncio
import cv2
import numpy as np
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js

# JavaScript Bridge untuk Streaming Webcam Browser ke Google Colab
LIVE_WEBCAM_JS = """
async function initTeleopWebcam() {
  const existing = document.getElementById('teleop-webcam-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'teleop-webcam-box';
  container.style.fontFamily = 'monospace';
  container.style.background = '#1a1a1a';
  container.style.color = '#00ffcc';
  container.style.padding = '14px';
  container.style.borderRadius = '10px';
  container.style.display = 'inline-block';
  container.style.boxShadow = '0 4px 12px rgba(0,0,0,0.5)';

  const header = document.createElement('div');
  header.innerHTML = '<span style="font-size:16px; font-weight:bold;">UR5 Gesture Teleoperation Stream</span>';
  header.style.marginBottom = '10px';
  container.appendChild(header);

  const video = document.createElement('video');
  video.id = 'teleop-video';
  video.style.borderRadius = '8px';
  video.style.transform = 'scaleX(-1)';
  video.width = 480;
  video.height = 360;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  container.appendChild(video);

  const hud = document.createElement('div');
  hud.id = 'teleop-hud';
  hud.style.marginTop = '8px';
  hud.style.fontSize = '13px';
  hud.innerHTML = 'Status: Meminta izin akses webcam...';
  container.appendChild(hud);

  const btnRow = document.createElement('div');
  btnRow.style.marginTop = '10px';
  const stopBtn = document.createElement('button');
  stopBtn.id = 'teleop-stop-btn';
  stopBtn.innerText = 'Stop Teleoperation Stream';
  stopBtn.style.background = '#e74c3c';
  stopBtn.style.color = '#fff';
  stopBtn.style.border = 'none';
  stopBtn.style.padding = '8px 16px';
  stopBtn.style.borderRadius = '5px';
  stopBtn.style.cursor = 'pointer';
  stopBtn.style.fontWeight = 'bold';

  stopBtn.onclick = () => {
    window.teleopStopped = true;
    window.teleopActive = false;
    if (window.teleopStream) {
      window.teleopStream.getTracks().forEach(track => track.stop());
    }
    stopBtn.innerText = '⏹️ Teleoperation Ended';
    stopBtn.disabled = true;
    stopBtn.style.background = '#7f8c8d';
    hud.innerHTML = 'Status: Sesi dihentikan.';
  };
  btnRow.appendChild(stopBtn);
  container.appendChild(btnRow);

  document.querySelector('#output-body').appendChild(container);

  window.teleopReady = false;
  window.teleopStopped = false;
  window.teleopActive = false;

  try {
    const stream = await navigator.mediaDevices.getUserMedia({video: {width: 480, height: 360}});
    window.teleopStream = stream;
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width = 480;
    canvas.height = 360;
    const ctx = canvas.getContext('2d');

    window.teleopActive = true;
    window.teleopReady = true;

    window.captureTeleopFrame = function() {
      if (!window.teleopActive || window.teleopStopped) return null;
      ctx.drawImage(video, 0, 0, 480, 360);
      return canvas.toDataURL('image/jpeg', 0.65);
    };

    window.updateHUD = function(gesture, conf, dwell, action) {
      const el = document.getElementById('teleop-hud');
      if (!el || window.teleopStopped) return;
      const pct = Math.round(conf * 100);
      const dwellPct = Math.round(dwell * 100);
      el.innerHTML = `Gesture: <b>${gesture}</b> (${pct}%) | Dwell: <b>${dwellPct}%</b> | Next Action: <b>${action || '-'}</b>`;
    };

    hud.innerHTML = 'Status: <span style="color:#2ecc71;">Kamera Aktif & Siap!</span>';
  } catch (err) {
    hud.innerHTML = `<span style="color:#e74c3c;">Akses kamera ditolak/gagal: ${err.message}</span>`;
    window.teleopReady = false;
    window.teleopStopped = true;
  }
}
initTeleopWebcam();
"""

def decode_base64_frame(b64_data: str) -> np.ndarray:
    if not b64_data:
        return None
    header, encoded = b64_data.split(',', 1) if ',' in b64_data else ('', b64_data)
    img_bytes = base64.b64decode(encoded)
    img_np = np.frombuffer(img_bytes, dtype=np.uint8)
    frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
    return cv2.flip(frame, 1) if frame is not None else None


async def start_live_gesture_teleop(duration_seconds: float = 90.0, conf_threshold: float = 0.50):
    """
    Loop teleoperasi real-time:
      Webcam -> MediaPipe -> ML Model -> Debounce Dwell Filter -> Physical UR5 Execution
    """
    # 1. Pastikan memiliki kunci eksklusif robot untuk sesi teleoperasi
    if not session.token:
        await session.acquire_lock(duration_s=duration_seconds + 15.0)

    # 2. Inisiasi widget kamera di browser
    display(Javascript(LIVE_WEBCAM_JS))

    print("Menunggu izin kamera dari browser (klik 'Allow' jika diminta)...")
    camera_ready = False
    for _ in range(30):
        ready = eval_js('window.teleopReady ? true : false')
        stopped = eval_js('window.teleopStopped ? true : false')
        if ready:
            camera_ready = True
            break
        if stopped:
            break
        await asyncio.sleep(0.5)

    if not camera_ready:
        print("Kamera tidak dapat diakses atau izin ditolak oleh browser.")
        print("Tips: Klik icon gembok di URL bar browser Colab -> izinkan Kamera (Camera: Allow), lalu jalankan ulang cell.")
        if session.token:
            await session.release_lock()
        return

    print("Live Teleoperation Aktif! Arahkan gestur tangan Anda ke kamera.")
    print("Tahan gestur selama 1.0 detik untuk memicu gerakan lengan UR5.")
    print("Klik tombol [Stop Teleoperation Stream] pada widget kamera untuk berhenti.\\n")

    start_time = time.time()

    try:
        while (time.time() - start_time) < duration_seconds:
            # Periksa apakah user mengklik tombol Stop
            user_stopped = eval_js('window.teleopStopped ? true : false')
            if user_stopped:
                print("⏹️ Sesi teleoperasi dihentikan oleh pengguna.")
                break

            # A. Ambil frame JPEG base64 dari browser
            frame_b64 = eval_js('window.captureTeleopFrame ? window.captureTeleopFrame() : null')
            if frame_b64 is None:
                await asyncio.sleep(0.05)
                continue

            # B. Dekode frame gambar
            frame = decode_base64_frame(frame_b64)
            if frame is None:
                await asyncio.sleep(0.05)
                continue

            # C. Deteksi landmark MediaPipe
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands_detector.process(rgb)

            current_gesture = "NONE"
            current_conf = 0.0
            dwell_progress = 0.0
            mapped_action = None

            if results.multi_hand_landmarks:
                hand_lms = results.multi_hand_landmarks[0]
                feats = extract_normalized_features(hand_lms)

                if feats is not None:
                    # D. Prediksi gesture via Random Forest
                    current_gesture, current_conf = predict_gesture_with_confidence(feats)
                    mapped_action = GESTURE_TO_ACTION.get(current_gesture)

                    # E. Filter kestabilan dwell
                    if current_conf >= conf_threshold:
                        is_triggered = debouncer.update(current_gesture)
                        dwell_progress = debouncer.get_progress()

                        if is_triggered and mapped_action:
                            print(f"\\nGESTURE TRIGGER TERVERIFIKASI: [{current_gesture}] ({current_conf*100:.1f}%) -> Eksekusi [{mapped_action}]!")
                            # Eksekusi aksi ke robot fisik
                            await handle_gesture_trigger(current_gesture)

            # F. Perbarui visual HUD di browser
            safe_act = mapped_action if mapped_action else "-"
            eval_js(f'window.updateHUD ? window.updateHUD("{current_gesture}", {current_conf}, {dwell_progress}, "{safe_act}") : null')

            await asyncio.sleep(0.04)  # ~25 FPS pacing loop

    except asyncio.CancelledError:
        print("Teleoperation task dibatalkan.")
    except Exception as e:
        print(f"Exception dalam teleop loop: {e}")
    finally:
        # Cleanup browser camera track
        eval_js('if (window.teleopStream) { window.teleopStream.getTracks().forEach(t => t.stop()); window.teleopActive = false; window.teleopStopped = true; }')
        if session.token:
            await session.release_lock()
        print("Sesi teleoperasi selesai. Kunci robot dilepaskan aman.")


# Jalankan live teleoperasi kamera (default aktif selama 90 detik)
await start_live_gesture_teleop(duration_seconds=90.0, conf_threshold=0.50)


## 3.5 Rangkuman Praktikum Manipulator UR5

Selamat, Anda telah berhasil menyelesaikan seluruh rangkaian praktikum teleoperasi robot manipulator UR5:
- [x] Sesi komunikasi WebSocket terhubung dan sistem sewa kunci (mutex lock) berfungsi.
- [x] Fitur 42-geometri dinormalisasi secara invarian translasi dan skala.
- [x] Dataset 8 gesture terkumpul dengan seimbang dan model Random Forest berhasil dilatih.
- [x] Pemetaan kuadran catur tervalidasi dalam jangkauan aman meja robot.
- [x] Filter Dwell Debouncer (1.0s) berhasil menyaring kedipan deteksi dan mencegah perintah berulang.
- [x] Siklus Pick and Place berhasil dieksekusi ke robot fisik/simulasi di Laboratorium B401.


# Bab 4: Interaksi dan Kendali Robot Berkaki Empat (Quadruped RoboDog)

Pada Bab 4, mahasiswa mengeksplorasi interaksi alami dengan robot berkaki empat (*Quadruped Robot Dog*). Modul ini memanfaatkan deteksi pose tubuh (*body pose tracking*) dan gestur dinamis untuk mengendalikan sikap tubuh (*stance*), arah hadap (*orientation*), dan navigasi robot di lingkungan nyata.

---

## 4.1 Persiapan dan Koneksi API RoboDog

Sebelum memulai interaksi, kita menghubungkan Google Colab dengan antarmuka pemrograman aplikasi (API) RoboDog melalui jaringan lab.


In [ ]:
# --- TULIS KONFIGURASI KONEKSI API ROBODOG DI SINI ---
# [AREA KELOMPOK: MODUL ROBODOG]

ROBODOG_API_URL = "http://lab-pc.local:8080/api/robodog"

def connect_robodog():
    """Inisialisasi handshake dengan controller RoboDog."""
    print(f"Menghubungkan ke controller RoboDog di {ROBODOG_API_URL}...")
    # Tulis logika inisialisasi koneksi rekan mahasiswa di sini
    print("Koneksi RoboDog standby.")
    return True

connect_robodog()


## 4.2 Interaksi Berbasis Pose dan Gesture dengan RoboDog

Pada bagian ini, mahasiswa memetakan postur tubuh (seperti kedua tangan terangkat, jongkok, atau gestur tangan tertentu) menjadi perintah sikap (*posture modes*) RoboDog:
* Pose Berdiri Tegak $\to$ Robot Mode `STAND`
* Pose Merendah / Jongkok $\to$ Robot Mode `CROUCH` / `SIT`
* Gestur Melambai $\to$ Respon Gerak Sapaan (*Greeting Motion*)


In [ ]:
# --- IMPLEMENTASI PEMETAAN POSE KE AKSI ROBODOG DI SINI ---
# [AREA KELOMPOK: MODUL ROBODOG]

def map_human_pose_to_robodog(pose_landmarks):
    """
    Menerima landmark tubuh (misal: MediaPipe Pose 33-titik)
    dan mengembalikan perintah sikap RoboDog.
    """
    # Tulis logika klasifikasi postur tubuh di sini
    return "STAND"

print("Modul pemetaan pose RoboDog siap dikembangkan.")


## 4.3 Navigasi dan Respon Gerak RoboDog

Mengirimkan instruksi kecepatan linear $(v_x, v_y)$ dan kecepatan sudut $(\omega_z)$ ke motor sendi kaki RoboDog secara terkoordinasi.


In [ ]:
# --- IMPLEMENTASI PENGIRIMAN PERINTAH NAVIGASI ROBODOG ---
# [AREA KELOMPOK: MODUL ROBODOG]

def dispatch_robodog_motion(vx: float, vy: float, omega: float, duration_s: float = 1.0):
    """Mengirim perintah gerak navigasi ke RoboDog."""
    print(f"RoboDog bergerak: vx={vx:.2f} m/s, vy={vy:.2f} m/s, omega={omega:.2f} rad/s")
    # Tulis pemanggilan API pengiriman gerak di sini
    return True

# dispatch_robodog_motion(0.5, 0.0, 0.0)


## 4.4 Tantangan dan Studi Kasus Terpadu (Challenges)

### Tantangan Kolaborasi Multi-Robot:
Rancang skenario di mana **RoboDog** mendeteksi keberadaan objek, mendekatinya, lalu memberikan sinyal ke **Robot Manipulator UR5** untuk mengambil objek tersebut dan menempatkannya ke dalam wadah penyimpanan.

---

## Daftar Periksa Akhir Workshop Human-Robot Interaction
- [x] Bab 0: Persiapan dependensi dan komunikasi sistem.
- [x] Bab 1: Rekayasa fitur pose-posture dan model Machine Learning.
- [ ] Bab 2: Perintah tingkat tinggi UR5 menggunakan LLM.
- [x] Bab 3: Kendali teleoperasi waypoint manipulator UR5.
- [ ] Bab 4: Interaksi dan respon gerak RoboDog.
